# 🚀 学生面板 | Student Dashboard
## 智能电动汽车（IEV）个人学习驾驶舱与工程自检中心
> 🌟 **版本认证**：**IEV CloudLab V1.0 官方标准工具 (Official Standard Tool)**
> 💡 **功能导视**：
> 本面板专为学生打造，与《教师面板》互补联动。在这里，您可以一站式观测自身学习进展、实时诊断七维工程能力雷达图、使用 IEV Copilot 智能导师进行第一性原理调参、提交前自检实验报告防伪指标，并在学期结业时一键导出工程成果包 (ZIP)！


In [ ]:
# 1. 环境初始化与依赖加载
import sys, os, json
for p in ["/home/jovyan/labs", os.path.abspath("../labs"), os.path.abspath("labs")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p); break

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from common import student_cockpit
from common import skills_profiler
from common import copilot
from common import semester_archiver
from common import report_evaluator
from common.evaluator import GRADING_SPECS

print("🚀 学生面板环境就绪！请在下方各交互功能区点击按钮体验。")


---
### 一、我的学习全景与七维工程能力雷达图 | Study Progress & Radar
点击下方按钮，系统将自动扫描您在  下生成的所有仿真结果，核算 29 门实验达成度并动态生成 7 维工程能力画像与攻坚推荐：


In [ ]:
out_progress = widgets.Output()
btn_refresh_progress = widgets.Button(description="🔄 刷新我的进度与能力画像", button_style="primary")

def on_refresh_progress(_):
    with out_progress:
        clear_output()
        progress = student_cockpit.get_student_study_progress()
        username = os.environ.get("JUPYTERHUB_USER", "同学")
        display(HTML(student_cockpit.render_student_cockpit_html(progress, student_name=username)))

btn_refresh_progress.on_click(on_refresh_progress)
display(btn_refresh_progress, out_progress)
on_refresh_progress(None)


---
### 二、29 门实验快速导航与直达通道 | Quick Experiment Launcher
从下拉列表中任选一门实验，即可快速查看核心指标判据标准、当前完成状态并一键直达实验 Notebook：


In [ ]:
out_launcher = widgets.Output()

exp_list = [
    ("A1 Cut-in加塞避险 (AEB制动)", "A1_cutin_aeb_pro.ipynb", "★ 入门", "最小TTC>1.5s 且无碰撞"),
    ("A2 ACC自适应巡航 (PD跟车)", "A2_acc_follow_pro.ipynb", "★ 入门", "最大间距误差≤1.5m"),
    ("A3 前车急刹避险 (CCRb增强TTC)", "A3_aeb_closedloop_pro.ipynb", "★★ 进阶", "增强版无碰撞 且误触发率<5%"),
    ("A4 交叉口左转让行 (LTAP-OD)", "A4_intersection_yield_pro.ipynb", "★★ 进阶", "PET>1.0s 且无碰撞"),
    ("A5 匝道汇入协同规划 (运动学)", "A5_merge_kinematic_demo_pro.ipynb", "★ 入门", "完整运行无异常"),
    ("B1 IDM跟车模型参数标定", "B1_idm_calibration_pro.ipynb", "★★ 进阶", "标定RMSE<0.6"),
    ("B2 交叉口信号配时优化 (Webster)", "B2_traffic_signal_pro.ipynb", "★ 入门", "平均延误优化率>10%"),
    ("B3 SUMO CACC 4车协同巡航", "B3_sumo_cacc_pro.ipynb", "★★ 进阶", "弦稳定性比值<1.0"),
    ("B4 拥堵传播分析 (LWR激波)", "B4_congestion_wave_pro.ipynb", "★★ 进阶", "冲击波波速吻合度>85%"),
    ("B5 干线绿波协调控制", "B5_green_wave_pro.ipynb", "★★★ 综合", "干线平均停车次数≤1次"),
    ("C1 FASTSim整车纵向电耗仿真", "C1_fastsim_energy_pro.ipynb", "★ 入门", "UDDS百公里电耗11~15 kWh"),
    ("C2 多车型横向能耗对比", "C2_vehicle_comparison_pro.ipynb", "★ 入门", "多车型排序与实测规律吻合"),
    ("C3 载重与风阻灵敏度分析", "C3_sensitivity_analysis_pro.ipynb", "★★ 进阶", "弹性回归判定主导效应"),
    ("C4 插电混动能量管理对比 (CD/CS)", "C4_ems_comparison_pro.ipynb", "★★ 进阶", "等效油耗对比合理"),
    ("D1 PyBaMM SPM单颗粒电池模型", "D1_pybamm_battery_pro.ipynb", "★ 入门", "1C放电容量相对误差<3%"),
    ("D2 倍率容量衰减特性 (Peukert)", "D2_rate_capacity_pro.ipynb", "★★ 进阶", "Peukert常数k在1.01~1.15"),
    ("D3 开路电压OCV-SOC曲线标定", "D3_ocv_soc_export_pro.ipynb", "★★ 进阶", "多倍率准静态拟合误差<1%"),
    ("D4 快充析锂风险电化学机理 (DFN)", "D4_lithium_plating_pro.ipynb", "★★★ 综合", "负极电位时空演变分析完整"),
    ("E1 单目标参数寻优 (减速器速比)", "E1_single_objective_pro.ipynb", "★ 入门", "工况百公里电耗显著降低"),
    ("E2 多目标Pareto前沿 (动力与经济)", "E2_pymoo_multiobj_pro.ipynb", "★★ 进阶", "NSGA-II非劣解集收敛良好"),
    ("E3 模型预测控制 (MPC前向跟驰)", "E3_mpc_follow_pro.ipynb", "★★★ 综合", "QP二次规划实时求解无碰撞"),
    ("E4 混动能量管理动态规划 (DP)", "E4_dp_ems_pro.ipynb", "★★★ 综合", "全局最优油耗优于基线策略"),
    ("F1 FMU模型导入与Co-Simulation", "F1_fmu_load_pro.ipynb", "★ 入门", "FMPy加载与步长仿真无报错"),
    ("F2 OpenModelica建模与FMU编译", "F2_openmodelica_fmu_pro.ipynb", "★★ 进阶", "与解析解相对误差<1e-3"),
    ("F3 多FMU跨域协同仿真 (车×电)", "F3_multi_fmu_pro.ipynb", "★★★ 综合", "双从站主算法通信步长稳定"),
    ("G1 ROS 2 话题发布与订阅 (DDS)", "G1_ros_topics_pro.ipynb", "★ 入门", "里程计与控制话题收发正常"),
    ("G2 ROS 2 服务通信与参数服务", "G2_ros_services_pro.ipynb", "★★ 进阶", "服务请求/响应响应时间<50ms"),
    ("H1 参数化场景批量生成与测试", "H1_batch_generation_pro.ipynb", "★★ 进阶", "全因子实验网格生成完整"),
    ("H2 场景模糊测试 (Fuzzing/SOTIF)", "H2_scenario_fuzzing_pro.ipynb", "★★★ 综合", "边界冲突变异算子检出率>80%"),
    ("🏆 期末跨域一体化综合大作业", "Capstone_iev_integrated_pro.ipynb", "★★★ 综合", "电池-整车-控制全链路闭环")
]

dropdown_exp = widgets.Dropdown(
    options=[(item[0], item[1]) for item in exp_list],
    value=exp_list[0][1],
    description="选择实验:",
    layout=widgets.Layout(width="460px")
)

def update_launcher_view(nb_filename):
    with out_launcher:
        clear_output()
        info = next((item for item in exp_list if item[1] == nb_filename), None)
        if not info: return
        exp_id = nb_filename.replace("_pro.ipynb", "").replace(".ipynb", "")
        res_candidates = [
            f"/home/jovyan/labs/data/results/{exp_id}.json",
            os.path.join("labs", "data", "results", f"{exp_id}.json"),
            os.path.join("..", "labs", "data", "results", f"{exp_id}.json"),
        ]
        is_done = any(os.path.exists(p) for p in res_candidates)
        status_badge = "<span style='color: #10b981; font-weight: bold;'>🟢 已完成</span>" if is_done else "<span style='color: #f59e0b; font-weight: bold;'>🟡 待完成</span>"
        
        display(HTML(f"""
        <div style="background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 14px 18px; margin-top: 10px;">
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <h4 style="margin: 0; color: #1e293b;">{info[0]}</h4>
                <div>{status_badge} <span style="background: #e2e8f0; color: #475569; padding: 2px 8px; border-radius: 4px; font-size: 12px; margin-left: 8px;">{info[2]}</span></div>
            </div>
            <p style="margin: 8px 0 10px 0; font-size: 13px; color: #475569;"><strong>判据标准：</strong>{info[3]}</p>
            <a href="./{info[1]}" target="_blank" style="display: inline-block; background: #2563eb; color: #ffffff; padding: 6px 16px; border-radius: 6px; text-decoration: none; font-size: 13px; font-weight: 600;">
                🚀 打开实验 Notebook 进行实操 ↗
            </a>
        </div>
        """))

def on_exp_change(change):
    if change["type"] == "change" and change["name"] == "value":
        update_launcher_view(change["new"])

dropdown_exp.observe(on_exp_change)
display(dropdown_exp, out_launcher)
update_launcher_view(dropdown_exp.value)


---
### 三、IEV Copilot 智能工程导师·调参诊断答疑 | Copilot Diagnostics
在实验过程中遇到指标未达标或仿真失稳？选择目标实验工况并点击诊断，Copilot 将为您提供**第一性原理物理归因、经验参数区间与排错指引**：


In [ ]:
out_copilot = widgets.Output()
copilot_exp_dd = widgets.Dropdown(
    options=[
        ("A1 加塞避险 (AEB碰撞 / TTC过小)", "A1_cutin_aeb"),
        ("A3 前车急刹 (CCRb 增强TTC误触发)", "A3_aeb_closedloop"),
        ("B1 IDM 跟车模型标定 (RMSE偏大)", "B1_idm_calibration"),
        ("C1 FASTSim 整车能耗 (电耗偏高)", "C1_fastsim_energy"),
        ("D4 快充析锂风险机理 (负极过电位越界)", "D4_lithium_plating"),
        ("E3 MPC 自适应跟驰 (QP求解超时/失稳)", "E3_mpc_follow"),
        ("E4 混动能量管理 DP (逆向递推违约)", "E4_dp_ems")
    ],
    value="A1_cutin_aeb",
    description="诊断工况:",
    layout=widgets.Layout(width="400px")
)
cb_collision = widgets.Checkbox(value=True, description="触发危险边界 / 约束越界")
btn_ask_copilot = widgets.Button(description="🤖 获取物理机理诊断与调参处方", button_style="info")

def on_ask_copilot(_):
    with out_copilot:
        clear_output()
        exp = copilot_exp_dd.value
        mock_metrics = {"collision": cb_collision.value, "min_ttc_s": 0.85 if cb_collision.value else 2.1}
        copilot.ask_copilot(exp, mock_metrics)

btn_ask_copilot.on_click(on_ask_copilot)
display(widgets.HBox([copilot_exp_dd, cb_collision, btn_ask_copilot]), out_copilot)


---
### 四、实验报告提交前预审自检器 | Pre-Submission Report Audit
在将实验报告交给老师批改前，先在这里运行一键自检！系统将自动核对 Markdown 报告中的数值与您本地仿真输出的真实一致性，杜绝笔误与抄写失误：


In [ ]:
out_report_check = widgets.Output()
txt_report_name = widgets.Text(
    value="实验报告模板_示例.md",
    description="报告文件:",
    placeholder="如 实验报告模板_示例.md 或 A1_report.md",
    layout=widgets.Layout(width="360px")
)
btn_check_report = widgets.Button(description="🛡️ 提交前一键自检自评", button_style="warning")

def on_check_report(_):
    with out_report_check:
        clear_output()
        rep_p = txt_report_name.value.strip()
        student_cockpit.pre_submit_report_check(rep_p)

btn_check_report.on_click(on_check_report)
display(widgets.HBox([txt_report_name, btn_check_report]), out_report_check)


---
### 五、学期结课作品集与工程数字证书导出 | Graduation Portfolio & Certificate
学期结课时，点击下方按钮一键打包您的个人成果。系统将自动汇总您的 29 个实验代码快照、仿真产物 JSON、高精度矢量能力雷达图与结业认证证书：


In [ ]:
out_portfolio = widgets.Output()
btn_export_portfolio = widgets.Button(description="📦 一键打包导出我的工程作品集 (ZIP)", button_style="success")

def on_export_portfolio(_):
    with out_portfolio:
        clear_output()
        print("正在聚合个人 29 门实验代码快照、仿真产物与工程能力雷达图...")
        res = semester_archiver.export_student_portfolio()
        if res.get("success"):
            zip_p = res.get("zip_path", "")
            print(f"✓ 专属工程作品集打包完成: {zip_p}")
            display(HTML(semester_archiver.render_download_link_html(zip_p, "📥 点击下载我的 IEV 结课成就包 (ZIP)")))
        else:
            print("导出失败:", res.get("error"))

btn_export_portfolio.on_click(on_export_portfolio)
display(btn_export_portfolio, out_portfolio)
